In [1]:
import nfl_data_py as nfl
import pandas as pd
import os
from pathlib import Path
import asyncio
import nbformat
from nbclient import NotebookClient

week = 18
current_year = 2025

In [2]:
if hasattr(asyncio, "WindowsSelectorEventLoopPolicy"):
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

notebooks_dir = Path.cwd()
notebooks_to_run = ["wrs_rec_tds.ipynb", "wrs_rec_yds.ipynb", "wrs_receptions.ipynb"]

Path(f"../outputs/{current_year}").mkdir(parents=True, exist_ok=True)

for name in notebooks_to_run:
    path = notebooks_dir / name
    nb = nbformat.read(path, as_version=4)
    client = NotebookClient(
        nb,
        timeout=3600,
        kernel_name="python3",
        resources={"metadata": {"path": str(notebooks_dir)}},
    )
    print(f"Running {name}...")
    client.execute()
    nbformat.write(nb, path)
    print(f"Finished {name}")

Running wrs_rec_tds.ipynb...


Finished wrs_rec_tds.ipynb
Running wrs_rec_yds.ipynb...


Finished wrs_rec_yds.ipynb
Running wrs_receptions.ipynb...


Finished wrs_receptions.ipynb


In [3]:
rec_yds = pd.read_csv(f"..\outputs\{current_year}\wrs_rec_yds_pred_week_{week}.csv")
receptions = pd.read_csv(f"..\outputs\{current_year}\wrs_receptions_pred_week_{week}.csv")
rec_tds = pd.read_csv(f"..\outputs\{current_year}\wrs_rec_tds_pred_week_{week}.csv")
first = pd.merge(rec_yds, receptions, on=['player_id', 'display_name'])
first = first[['player_id', 'display_name', 'predicted_receptions', 'predicted_receiving_yards', 'receiving_yards_season_avg', 'receptions_season_avg']]
second = pd.merge(first, rec_tds, on=['player_id', 'display_name'])
final = second[['player_id', 'display_name',
                'predicted_receptions', 'predicted_receiving_yards', 'predicted_receiving_tds',
                'receiving_yards_season_avg', 'receptions_season_avg', 'receiving_tds_season_avg']]
final['projected_std_points'] = round(0.1*final['predicted_receiving_yards'] + 6*final['predicted_receiving_tds'], 4)
final['projected_ppr_points'] = round(1*final['predicted_receptions'] + 0.1*final['predicted_receiving_yards'] + 6*final['predicted_receiving_tds'], 4)

C:\Users\rille\AppData\Local\Temp\ipykernel_6580\1976404465.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final['projected_std_points'] = round(0.1*final['predicted_receiving_yards'] + 6*final['predicted_receiving_tds'], 4)
C:\Users\rille\AppData\Local\Temp\ipykernel_6580\1976404465.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final['projected_ppr_points'] = round(1*final['predicted_receptions'] + 0.1*final['predicted_receiving_yards'] + 6*final['predicted_receiving_tds'], 4)


In [4]:
ordered = final.sort_values(by='projected_ppr_points', ascending=False)
ordered.to_csv(f"..\outputs\{current_year}\wrs_complete_week{week}.csv")
ordered.head(10)

,player_id,display_name,predicted_receptions,predicted_receiving_yards,predicted_receiving_tds,receiving_yards_season_avg,receptions_season_avg,receiving_tds_season_avg,projected_std_points,projected_ppr_points
0,00-0036900,Ja'Marr Chase,7.462908,92.645250,0.352275,87.733333,8.666667,0.466667,11.3782,18.8411
16,00-0038543,Jaxon Smith-Njigba,6.665025,82.328240,0.645641,106.812500,5.666667,0.600000,12.1067,18.7717
2,00-0036900,Ja'Marr Chase,7.306407,92.645250,0.352275,87.733333,8.000000,0.466667,11.3782,18.6846
20,00-0038543,Jaxon Smith-Njigba,6.665025,79.813736,0.645641,99.600000,5.666667,0.600000,11.8552,18.5202
18,00-0038543,Jaxon Smith-Njigba,6.245005,82.328240,0.645641,106.812500,8.000000,0.600000,12.1067,18.3517
1,00-0036900,Ja'Marr Chase,7.462908,92.645250,0.229656,87.733333,8.666667,0.500000,10.6425,18.1054
22,00-0038543,Jaxon Smith-Njigba,6.245005,79.813736,0.645641,99.600000,8.000000,0.600000,11.8552,18.1002
3,00-0036900,Ja'Marr Chase,7.306407,92.645250,0.229656,87.733333,8.000000,0.500000,10.6425,17.9489
4,00-0036900,Ja'Marr Chase,7.462908,82.346810,0.352275,88.250000,8.666667,0.466667,10.3483,17.8112
32,00-0036358,CeeDee Lamb,5.977429,77.605390,0.672441,89.416667,5.666667,0.250000,11.7952,17.7726
